# pass@k: hands-on (optional)
**LLMs, Animated · Lesson M4.2**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab).

In [ ]:
import math, itertools
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
print("ready")

## 1 · The unbiased estimator
For one problem you drew $n$ samples and $c$ passed the tests. Chen et al. (2021) estimate pass@k as

$$1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}$$

the chance that a random pick of $k$ of the $n$ samples contains at least one pass. Use `math.comb`. Remember: if $n - c < k$, every pick contains a pass.

In [ ]:
def pass_at_k(n, c, k):
    # TODO
    raise NotImplementedError("Exercise 1")

**Check 1.** The numbers from the lesson.

In [ ]:
assert abs(pass_at_k(10, 3, 1) - 0.3) < 1e-12, "for k = 1 it's just c/n"
assert abs(pass_at_k(10, 3, 5) - (1 - 21 / 252)) < 1e-12
assert abs(pass_at_k(10, 2, 3) - (1 - 56 / 120)) < 1e-12
assert pass_at_k(5, 1, 5) == 1.0
assert pass_at_k(10, 0, 3) == 0.0
print("✓ exercise 1: pass@5 =", round(pass_at_k(10, 3, 5), 4))

## 2 · Check it by brute force
Make the definition concrete: list all $\binom{n}{k}$ ways to pick $k$ of the $n$ samples with `itertools.combinations`, and return the fraction of picks that contain at least one pass.

In [ ]:
def pass_at_k_brute(results, k):
    # results: list of booleans, True = that sample passed
    # TODO
    raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
for n, c, k in [(10, 3, 5), (8, 1, 3), (7, 4, 2), (6, 0, 2), (9, 2, 7)]:
    res = [True] * c + [False] * (n - c)
    assert abs(pass_at_k_brute(res, k) - pass_at_k(n, c, k)) < 1e-12, (n, c, k)
print("✓ exercise 2: the formula counts exactly the picks that contain a pass")

## 3 · A numerically stable version
With $n = 200$ the binomials have dozens of digits. Implementations use the equivalent product

$$\frac{\binom{n-c}{k}}{\binom{n}{k}} = \prod_{i=n-c+1}^{n}\left(1 - \frac{k}{i}\right)$$

Implement it with NumPy (`np.prod`, `np.arange`).

In [ ]:
def pass_at_k_stable(n, c, k):
    if n - c < k:
        return 1.0
    # TODO
    raise NotImplementedError("Exercise 3")

**Check 3**

In [ ]:
for _ in range(300):
    n = int(rng.integers(1, 201)); c = int(rng.integers(0, n + 1)); k = int(rng.integers(1, n + 1))
    assert abs(pass_at_k_stable(n, c, k) - pass_at_k(n, c, k)) < 1e-9, (n, c, k)
print("✓ exercise 3, e.g. n=200, c=7: pass@1 =", round(pass_at_k_stable(200, 7, 1), 4), " pass@100 =", round(pass_at_k_stable(200, 7, 100), 4))

## 4 · Why not just plug in c/n?
The tempting shortcut is $1 - (1 - c/n)^k$. Let's test both estimators on a simulated benchmark where we *know* the truth.

- 500 problems, each with a true pass rate $p_i$ drawn from a Beta distribution (many hard problems, a few easy ones).
- True pass@k of the benchmark: the average of $1 - (1 - p_i)^k$.
- One "evaluation run": draw $c_i \sim \text{Binomial}(n, p_i)$ for every problem and average each estimator over problems.

Complete `one_run`, which returns `(unbiased_estimate, shortcut_estimate)` for one run.

In [ ]:
P_TRUE = rng.beta(0.5, 2.0, size=500)
N, K = 10, 5
true_pass_k = np.mean(1 - (1 - P_TRUE) ** K)
print("true pass@5 of this benchmark:", round(true_pass_k, 4))

In [ ]:
def one_run(p_true, n, k, rng):
    c = rng.binomial(n, p_true)       # passes per problem in this run
    # TODO: average pass_at_k over problems, and average the shortcut 1-(1-c/n)^k
    raise NotImplementedError("Exercise 4")

**Check 4.** Over 200 runs, the unbiased estimator averages out to the truth; the shortcut is systematically low.

In [ ]:
runs = np.array([one_run(P_TRUE, N, K, rng) for _ in range(200)])
mean_unb, mean_short = runs.mean(axis=0)
print(f"true {true_pass_k:.4f} | unbiased {mean_unb:.4f} | shortcut {mean_short:.4f}")
assert abs(mean_unb - true_pass_k) < 0.006
assert mean_short < true_pass_k - 0.02
print("✓ exercise 4: the shortcut underestimates pass@k (Jensen's inequality: 1-(1-p)^k is concave in p)")

## Experiment
1. How does the shortcut's bias depend on $n$? Run the simulation for $n \in \{5, 10, 20, 50, 200\}$ (with $k = 5$) and plot the bias of each estimator. The shortcut's bias shrinks as $n$ grows, but the unbiased one has none to begin with. That's one reason Chen et al. used $n = 200$.
2. Try $k = 1$: are the two estimators different there?

In [ ]:
ns = [5, 10, 20, 50, 200]
bias_u, bias_s = [], []
for n in ns:
    r = np.array([one_run(P_TRUE, n, K, rng) for _ in range(60)]).mean(axis=0)
    bias_u.append(r[0] - true_pass_k); bias_s.append(r[1] - true_pass_k)
plt.figure(figsize=(7, 4))
plt.axhline(0, color="gray", lw=0.8)
plt.plot(ns, bias_u, "o-", label="unbiased estimator"); plt.plot(ns, bias_s, "o--", label="shortcut 1-(1-c/n)^k")
plt.xscale("log"); plt.xlabel("samples per problem n"); plt.ylabel("bias of pass@5 estimate"); plt.legend()
plt.title("The shortcut is biased low, less so with more samples"); plt.show()

## Recap
- pass@k = probability that at least one of k samples passes; estimate it per problem with $1 - \binom{n-c}{k}/\binom{n}{k}$ and average.
- It equals the fraction of k-subsets containing a pass (you checked by brute force).
- Use the product form for large n.
- The plug-in shortcut $1-(1-c/n)^k$ is biased low; the bias shrinks with more samples.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    return 1 - math.comb(n - c, k) / math.comb(n, k)

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def pass_at_k_brute(results, k):
    picks = list(itertools.combinations(results, k))
    return sum(any(p) for p in picks) / len(picks)

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def pass_at_k_stable(n, c, k):
    if n - c < k:
        return 1.0
    return float(1 - np.prod(1 - k / np.arange(n - c + 1, n + 1)))

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def one_run(p_true, n, k, rng):
    c = rng.binomial(n, p_true)
    unbiased = np.mean([pass_at_k(n, int(ci), k) for ci in c])
    shortcut = np.mean(1 - (1 - c / n) ** k)
    return unbiased, shortcut